# NN Ensembler and Param Testbed

In [1]:
import os
if 'experiments' in os.getcwd ():
    os.chdir (os.getcwd () + "/..")

import numpy as np
import pandas as pd
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings ('ignore', category = ConvergenceWarning)

In [2]:
####### DATA PATHS #######
TRAIN_PATH = "./data/in/cattle_data_train.csv"
TEST_PATH = "./data/in/cattle_data_test.csv"
OUT_PATH = "./data/out/nnet_ensemble.csv"

# Preprocessing

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option ("display.max_columns", None)
TARGET_FEATURE = "Milk_Yield_L"

DROP_FEATURES = \
[
    "Cattle_ID",
    "Farm_ID",
    "Feed_Quantity_lb",
    "Breed",
    "Climate_Zone",
    "Management_System",
    "Feed_Type",
    "Feeding_Frequency",
    "Walking_Distance_km",
    "Grazing_Duration_hrs",
    "Rumination_Time_hrs",
    "Resting_Hours",
    "Body_Condition_Score",
    "Humidity_percent",
    "BVD_Vaccine",
    "FMD_Vaccine",
    "Brucellosis_Vaccine",
    "HS_Vaccine",
    "BQ_Vaccine",
    "Housing_Score"
]
        
CATEGORICAL_FEATURES = \
[
    "Lactation_Stage",
    "Date",
    "Milking_Interval_hrs"
]

STANDARD_SCALED_FEATURES = \
[
    "Age_Months",
    "Weight_kg",
    "Parity",
    "Days_in_Milk",
    "Feed_Quantity_kg",
    "Water_Intake_L",
    "Ambient_Temperature_C",
    "Previous_Week_Avg_Yield"
]

In [4]:
def month_to_season (
    m
) -> str:
    """
    Converts month number into season string
    """
    if m in [12, 1, 2]:
        return "Winter"
    elif m in [3, 4, 5]:
        return "Spring"
    elif m in [6, 7, 8]:
        return "Summer"
    else:
        return "Fall"


def preprocess (
    dtrain, dtest
) -> tuple[pd.DataFrame, pd.DataFrame, StandardScaler]:
    """
    Further cleaning and feature engineering
    Returns: [dtrain, dtest, scaler]
    """
    # Drop useless features 
    dtrain = dtrain.drop (DROP_FEATURES, axis = 1)
    dtest = dtest.drop (DROP_FEATURES, axis = 1)

    # Convert month to season
    months = pd.to_datetime (dtest['Date']).dt.month
    dtest = dtest.drop (columns = ['Date'])
    dtest['Date'] = months.apply (month_to_season)

    months = pd.to_datetime (dtrain['Date']).dt.month
    dtrain = dtrain.drop (columns = ['Date'])
    dtrain['Date'] = months.apply (month_to_season)

    # Imputation
    median_val = dtrain["Feed_Quantity_kg"].median ()

    dtrain.loc[dtrain["Feed_Quantity_kg"].isna (), "Feed_Quantity_kg"] = median_val
    dtest.loc[dtest["Feed_Quantity_kg"].isna (), "Feed_Quantity_kg"] = median_val

    # One-hot encode categoricals
    dtrain = pd.get_dummies (dtrain, columns = CATEGORICAL_FEATURES, 
                             drop_first = True)
    dtest = pd.get_dummies (dtest, columns = CATEGORICAL_FEATURES, 
                            drop_first = True)
    dtrain, dtest = dtrain.align (dtest, join = 'left', axis = 1, fill_value = 0)

    # Standardize numerics
    scaler = StandardScaler ()
    dtrain[STANDARD_SCALED_FEATURES] = scaler.fit_transform (dtrain[STANDARD_SCALED_FEATURES])
    dtest[STANDARD_SCALED_FEATURES] = scaler.transform (dtest[STANDARD_SCALED_FEATURES])

    return dtrain, dtest, scaler

In [5]:
# Feature engineer data
train_data = pd.read_csv (TRAIN_PATH)
X_train, X_test, y_train, y_test = train_test_split (
    train_data.drop (TARGET_FEATURE, axis = 1),
    train_data[TARGET_FEATURE], test_size = 0.2, random_state = 0)

X_train, X_test, scaler = preprocess (X_train, X_test)

# Training Ensemble

In [ ]:
# Ensemble configs with solver-specific params
ensemble_configs = [{'hidden_layer_sizes': (128, 128, 128), 
                     'alpha': 0.001, 
                     'activation': 'relu',
                     'learning_rate_init': 0.00004},

                     {'hidden_layer_sizes': (100, 100, 100)},]

# Train ensemble
models = []
train_rmse_list = []
test_rmse_list = []
best_test_rmse_list = []
best_iter_list = []

total_iterations = 250
iterations_per_step = 10

# Train all configs
print (f"Training ensemble of {len (ensemble_configs)} models...\n")
for idx, config in enumerate (ensemble_configs):
    print (f"Training model {idx+1}/{len (ensemble_configs)}")

    # Base common params
    base_params = dict (hidden_layer_sizes = config['hidden_layer_sizes'],
                        alpha              = config.get ('alpha', 0),
                        solver             = config.get ('solver', 'adam'),
                        learning_rate_init = config.get ("learning_rate_init", 0.00003),
                        activation         = config.get ('activation', 'tanh'),
                        max_iter           = iterations_per_step,
                        learning_rate      = "adaptive",
                        random_state       = 1,
                        warm_start         = True,
                        early_stopping     = False,
                        n_iter_no_change   = 20,
                        verbose            = False)

    solver_params = config.get ('solver_params', {})
    all_params = {**base_params, **solver_params}

    model = MLPRegressor (**all_params)

    # Track best RMSE
    best_test_rmse = float ('inf')
    best_iter = 0
    prev_rmse = float ('inf')
    test_rmse = 0
    TOLERANCE = 0.00005
    total_iterations = 0

    while (test_rmse + TOLERANCE < prev_rmse):
        if test_rmse > 0:
            prev_rmse = test_rmse
        model.fit (X_train, y_train)

        # Compute RMSE
        y_train_pred = model.predict (X_train)
        y_test_pred = model.predict (X_test)
        train_rmse = np.sqrt (mean_squared_error (y_train, y_train_pred))
        test_rmse = np.sqrt (mean_squared_error (y_test, y_test_pred))

        total_iterations += iterations_per_step
        print (f"Iteration {total_iterations}: Train RMSE={train_rmse:.5f}, Test RMSE={test_rmse:.5f}")

    # Final evaluation
    y_train_pred = model.predict (X_train)
    y_test_pred = model.predict (X_test)
    train_rmse = np.sqrt (mean_squared_error (y_train, y_train_pred))
    test_rmse = np.sqrt (mean_squared_error (y_test, y_test_pred))

    print(f"  Final - Train RMSE: {train_rmse:.4f}, Test RMSE: {test_rmse:.4f}")
    print()

    models.append (model)
    train_rmse_list.append (train_rmse)
    test_rmse_list.append (test_rmse)

# Ensemble predictions
print ("=" * 60)
print ("ENSEMBLE RESULTS")
print ("=" * 60) 

train_preds = np.array ([model.predict (X_train) for model in models])
test_preds = np.array ([model.predict (X_test) for model in models])

ensemble_train_pred = np.median (train_preds, axis = 0)
ensemble_test_pred = np.median (test_preds, axis = 0)

ensemble_train_rmse = np.sqrt (mean_squared_error (y_train, ensemble_train_pred))
ensemble_test_rmse = np.sqrt (mean_squared_error (y_test, ensemble_test_pred))

print (f"\nIndividual model performance:")
print (f"  Train RMSE - Best: {min (train_rmse_list):.4f}, Worst: {max (train_rmse_list):.4f}, Mean: {np.mean( train_rmse_list):.4f}")
print (f"  Test RMSE - Best: {min (test_rmse_list):.4f}, Worst: {max (test_rmse_list):.4f}, Mean: {np.mean( test_rmse_list):.4f}")

print (f"\nEnsemble performance:")
print (f"  Ensemble Train RMSE: {ensemble_train_rmse:.4f}")
print (f"  Ensemble Test RMSE:  {ensemble_test_rmse:.4f}")
print (f"  Improvement over best final individual: {min (test_rmse_list) - ensemble_test_rmse:.4f}")

Training ensemble of 2 models...

Training model 1/2
Iteration 10: Train RMSE=4.18641, Test RMSE=4.19003
Iteration 20: Train RMSE=4.16101, Test RMSE=4.17160
Iteration 30: Train RMSE=4.15241, Test RMSE=4.16740
Iteration 40: Train RMSE=4.14604, Test RMSE=4.16448
Iteration 50: Train RMSE=4.13910, Test RMSE=4.16028
Iteration 60: Train RMSE=4.13007, Test RMSE=4.15351
Iteration 70: Train RMSE=4.11984, Test RMSE=4.14612
Iteration 80: Train RMSE=4.11140, Test RMSE=4.14069
Iteration 90: Train RMSE=4.10465, Test RMSE=4.13721
Iteration 100: Train RMSE=4.09917, Test RMSE=4.13550
Iteration 110: Train RMSE=4.09441, Test RMSE=4.13445
Iteration 120: Train RMSE=4.09022, Test RMSE=4.13435


/home/osten/.virtualenvs/IPYNB/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


Iteration 130: Train RMSE=4.09040, Test RMSE=4.13491
  Final - Train RMSE: 4.0904, Test RMSE: 4.1349

Training model 2/2


# Analyze Ensemble Predictions

In [ ]:
# Simple stacking without cross-validation
meta_features_train = np.column_stack([model.predict(X_train) for model in models])
meta_features_test = np.column_stack([model.predict(X_test) for model in models])

meta_model = LinearRegression()
meta_model.fit (meta_features_train, y_train)

stacked_train_pred = meta_model.predict(meta_features_train)
stacked_test_pred = meta_model.predict(meta_features_test)

stacked_train_rmse = np.sqrt (mean_squared_error (y_train, stacked_train_pred))
stacked_test_rmse = np.sqrt (mean_squared_error (y_test, stacked_test_pred))

print (f"Stack performance:")
print (f"  Stack Train RMSE: {stacked_train_rmse:.4f}")
print (f"  Stack Test RMSE:  {stacked_test_rmse:.4f}")
print (f"  Improvement over best final individual: {min (test_rmse_list) - stacked_test_rmse:.4f}")

<>:14: SyntaxWarning: invalid escape sequence '\S'
<>:14: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_209052/3943892635.py:14: SyntaxWarning: invalid escape sequence '\S'
  print (f"\Stack performance:")


\Stack performance:
  Stack Train RMSE: 4.0889
  Stack Test RMSE:  4.1195
  Improvement over best final individual: -0.0104


In [ ]:
# # Analyze best and worst predictions from ensemble
# raw_data = pd.read_csv (TRAIN_PATH)

# errors = np.sqrt ((y_test - ensemble_test_pred) ** 2)
# df_results = X_test.copy ()
# scaled_part = df_results[STANDARD_SCALED_FEATURES]
# scaled_inverse = pd.DataFrame (scaler.inverse_transform (scaled_part),
#                                columns = STANDARD_SCALED_FEATURES,
#                                index = df_results.index)

# # Replace only those columns
# df_results[STANDARD_SCALED_FEATURES] = scaled_inverse
# df_results["y_true"] = y_test
# df_results["y_pred"] = ensemble_test_pred
# df_results["rmse"] = errors

# df_results = df_results.merge (raw_data,
#                                left_index = True,
#                                right_index = True,
#                                how = "left")

# print ("\nTop 5 BEST predictions:")
# print (df_results.nsmallest (5, "rmse"))

# print ("\nTop 5 WORST predictions:")
# print (df_results.nlargest (5, "rmse"))

# Final Model - Train on Full Dataset

In [ ]:
# # Build final ensemble on full training data
# train_data = pd.read_csv (TRAIN_PATH)
# test_data = pd.read_csv (TEST_PATH)

# X_train_full = train_data.drop (TARGET_FEATURE, axis = 1)
# y_train_full = train_data[TARGET_FEATURE]
# X_test_full = test_data

# X_train_full, X_test_full, scaler_full = preprocess (X_train_full, X_test_full)

# print (f"Training final ensemble on full dataset...")
# print ()

# final_models = []

# for idx, config in enumerate (ensemble_configs):
#     print (f"Training final model {idx+1}/{len (ensemble_configs)}")
    
#     base_params = dict (hidden_layer_sizes = config['hidden_layer_sizes'],
#                         alpha              = config.get ('alpha', 0.0001),
#                         solver             = config.get ('solver', 'adam'),
#                         learning_rate_init = config.get ("learning_rate_init", 0.00003),
#                         activation         = config.get ('activation', 'tanh'),
#                         max_iter           = iterations_per_step,
#                         learning_rate      = "adaptive",
#                         random_state       = 0,
#                         warm_start         = True,
#                         early_stopping     = False,
#                         n_iter_no_change   = 20,
#                         verbose            = False)

#     solver_params = config.get ('solver_params', {})
#     all_params = {**base_params, **solver_params}
#     model = MLPRegressor (**all_params)

#     model.fit (X_train_full, y_train_full)
#     final_models.append (model)
#     print (f"  Model {idx+1} trained successfully")

# print ("\nAll models trained!")

In [ ]:
# # Final Predictions - ensemble average
# final_preds = np.array([model.predict(X_test_full) for model in final_models])
# y_pred_final = final_preds.mean(axis=0)

# print(f"Ensemble prediction mean: {y_pred_final.mean():.4f}")
# print(f"Ensemble prediction std: {y_pred_final.std():.4f}")
# print(f"Prediction range: [{y_pred_final.min():.4f}, {y_pred_final.max():.4f}]")

# out_data = pd.DataFrame({
#     'Cattle_ID': np.arange(1, len(y_pred_final) + 1),
#     'Milk_Yield_L': y_pred_final
# })
# out_data.to_csv(OUT_PATH, index=False)
# print(f"\nPredictions saved to {OUT_PATH}")